# 3. Public Access — Azure Firewall, App Gateway + WAF, Front Door, DDoS

## Azure Firewall

Centralized, managed, stateful firewall. Deployed in a hub VNet with spokes routed through it.

### Firewall SKUs

| | Standard | Premium |
|-|----------|----------|
| L3-L7 filtering | ✅ | ✅ |
| FQDN filtering | ✅ | ✅ |
| Threat intelligence | ✅ | ✅ |
| **TLS inspection** | ❌ | ✅ |
| **IDPS** (intrusion detection) | ❌ | ✅ |
| **URL filtering** (full URL, not just FQDN) | ❌ | ✅ |
| **Web categories** | ❌ | ✅ |
| Cost | ~$912/month | ~$1,825/month |

### Firewall rule types (processed in order)

| Rule type | What it filters | Example |
|-----------|----------------|----------|
| **NAT rules** | Inbound DNAT (port forwarding) | Forward public port 443 → internal VM |
| **Network rules** | L3/L4 (IP, port, protocol) | Allow 10.0.0.0/8 → SQL port 1433 |
| **Application rules** | L7 FQDN/URL | Allow *.microsoft.com, block *.gambling.com |

## Before you run this notebook

1. Run `uv sync` from the lab folder (`security-certs/az-500/02-networking`).
2. In VS Code, click the **kernel picker** at the top-right of this notebook and choose the interpreter from `.venv` (the one created by `uv`).
3. If the kernel isn't listed, reload the window (`Cmd+Shift+P` then *Reload Window*).

No Docker or Azure subscription is needed — everything is simulated in plain Python so you can learn the concepts safely.

In [ ]:
# Simulate Azure Firewall rule evaluation
FW_RULES = {
    'nat': [
        {'name': 'WebInbound', 'src': '*', 'dst_port': 443, 'translated_ip': '10.0.1.10', 'translated_port': 443},
    ],
    'network': [
        {'name': 'AllowAppToSQL', 'src': '10.0.2.0/24', 'dst': '10.0.3.0/24', 'port': 1433, 'action': 'Allow'},
        {'name': 'AllowDNS',     'src': '10.0.0.0/8',   'dst': '*',           'port': 53,   'action': 'Allow'},
    ],
    'application': [
        {'name': 'AllowMicrosoft',  'src': '10.0.0.0/8', 'fqdn': '*.microsoft.com',  'protocol': 'HTTPS', 'action': 'Allow'},
        {'name': 'AllowGitHub',     'src': '10.0.0.0/8', 'fqdn': '*.github.com',     'protocol': 'HTTPS', 'action': 'Allow'},
        {'name': 'BlockGambling',   'src': '10.0.0.0/8', 'fqdn': '*.gambling.com',   'protocol': 'HTTPS', 'action': 'Deny'},
        {'name': 'AllowUbuntuAPT',  'src': '10.0.0.0/8', 'fqdn': 'archive.ubuntu.com', 'protocol': 'HTTP', 'action': 'Allow'},
    ],
}

print('=== Azure Firewall Rule Sets ===')
for rule_type, rules in FW_RULES.items():
    print(f'\n--- {rule_type.upper()} rules (processed {"first" if rule_type == "nat" else "second" if rule_type == "network" else "last"}) ---')
    for r in rules:
        if rule_type == 'nat':
            print(f'  {r["name"]}: *:{r["dst_port"]} → {r["translated_ip"]}:{r["translated_port"]}')
        elif rule_type == 'network':
            print(f'  {r["name"]}: {r["src"]} → {r["dst"]}:{r["port"]} [{r["action"]}]')
        else:
            print(f'  {r["name"]}: {r["src"]} → {r["fqdn"]} [{r["action"]}]')

print('\n💡 Processing order: NAT → Network → Application → Implicit deny')

## 🚫 Bad → ✅ Best: Exposing a web app to the internet

| Stage | Architecture | Weakness |
|-------|-------------|----------|
| 🚫 **Bad** | VM with public IP, no NSG | Any port reachable; app vulnerable to L7 attacks |
| ⚠️ **Okay** | VM + NSG allowing 443 only | No WAF, no DDoS mitigation, SSL ends on the VM |
| 👍 **Better** | App Gateway + WAF in front of private VMs | Regional only, no global latency benefit |
| ✅ **Best** | Front Door (WAF + CDN at edge) → App Gateway (WAF) → VMs with NSG | Defense-in-depth, global, DDoS Standard on public IPs |

The next cells simulate a WAF and a defense-in-depth pipeline so you can see the layers in action.

In [ ]:
# Toy WAF: detects a few OWASP Top-10 patterns in HTTP requests.
import re

OWASP_RULES = [
    ("SQLi-basic",        re.compile(r"(?i)(' or '1'='1|union\s+select|--\s|;\s*drop\s+table)")),
    ("XSS-script-tag",    re.compile(r"(?i)<script\b")),
    ("PathTraversal",     re.compile(r"(\.\./){2,}")),
    ("CmdInjection",      re.compile(r"(?i)(;|\|)\s*(whoami|cat\s+/etc/passwd|rm\s+-rf)")),
]

def waf_evaluate(request, mode="Prevention"):
    hits = [name for name, rx in OWASP_RULES if rx.search(request)]
    if not hits:
        return "allow", []
    if mode == "Detection":
        return "allow-but-logged", hits   # WAF records but does not block
    return "block", hits                  # Prevention mode blocks

REQUESTS = [
    "GET /products?id=42 HTTP/1.1",
    "GET /products?id=1' or '1'='1 HTTP/1.1",
    "POST /comments body=<script>alert(1)</script>",
    "GET /../../../../etc/passwd HTTP/1.1",
    "GET /search?q=hello HTTP/1.1",
    "GET /ping?host=8.8.8.8;cat /etc/passwd HTTP/1.1",
]

for mode in ("Detection", "Prevention"):
    print(f"=== WAF mode: {mode} ===")
    for r in REQUESTS:
        decision, hits = waf_evaluate(r, mode)
        icon = {"allow": "✅", "allow-but-logged": "⚠️ ", "block": "🚫"}[decision]
        hits_str = ','.join(hits) if hits else '—'
        print(f"  {icon} {decision:<18} hits={hits_str:<30}  {r[:60]}")
    print()

print("Detection mode is what you use to tune rules; Prevention is what you run in production.")

## Defense-in-depth: Front Door → App Gateway → VMs

Real workloads layer multiple controls so that a bypass at one layer is caught by the next.
The next cell walks a single HTTP request through each layer, printing what happens at each.

In [ ]:
# Simulate a request traversing the full public-access stack.
def front_door(req):
    # Edge WAF: handles volumetric + common exploits at the POP.
    if "<script" in req.lower() or "union select" in req.lower():
        return "🚫 blocked at Front Door WAF (edge)"
    if req.startswith("HTTP "):  # Non-HTTPS arriving at edge is redirected
        return "↪️  Front Door: redirect HTTP→HTTPS"
    return None  # forward

def app_gateway(req):
    # Regional WAF + URL routing. Catches origin-specific exploits.
    if "../" in req:
        return "🚫 blocked at App Gateway WAF (region)"
    if " /admin" in req and "internal-token" not in req:
        return "🚫 blocked at App Gateway: /admin requires header"
    return None

def nsg(src_ip):
    # Only App Gateway subnet may reach the backend pool.
    if not src_ip.startswith("10.0.5."):
        return f"🚫 blocked at NSG (src {src_ip} not in App Gateway subnet)"
    return None

def serve(req):
    return f"✅ 200 OK from backend VM ({req[:50]}...)"

def pipeline(req, appgw_src="10.0.5.4"):
    for layer, fn, arg in [
        ("Front Door", front_door, req),
        ("App Gateway", app_gateway, req),
        ("NSG",        nsg, appgw_src),
    ]:
        verdict = fn(arg)
        if verdict:
            return f"[{layer}] {verdict}"
    return serve(req)

REQUESTS = [
    "GET / HTTP/1.1",
    "GET /products?id=1 union select * from users HTTP/1.1",      # blocked at edge
    "POST /comments <script>alert(1)</script>",                   # blocked at edge
    "GET /../../../../etc/passwd HTTP/1.1",                       # blocked at App Gateway
    "GET /admin HTTP/1.1",                                        # missing header
    "GET /admin HTTP/1.1 internal-token=abc",                     # allowed
]

for r in REQUESTS:
    print(pipeline(r))

# Bonus: what if an attacker tries to hit the VM directly, skipping App Gateway?
print("\nDirect VM hit bypass attempt:")
print(pipeline("GET / HTTP/1.1", appgw_src="203.0.113.7"))

### Azure Firewall Manager

Manages firewall policies across multiple firewalls and regions:
- **Firewall policies** are hierarchical (parent → child inheritance)
- **Secured virtual hubs** integrate firewall with Virtual WAN
- Central management for global enterprises

```bash
# Create a firewall policy
az network firewall policy create -g rg-hub -n fw-policy-global \
  --sku Premium --threat-intel-mode Deny

# Create child policy inheriting from parent
az network firewall policy create -g rg-hub -n fw-policy-westeurope \
  --base-policy fw-policy-global --sku Premium
```

---
## Application Gateway + WAF

| Feature | Application Gateway | Azure Firewall |
|---------|-------------------|----------------|
| Layer | L7 (HTTP/HTTPS only) | L3-L7 (any protocol) |
| SSL termination | ✅ | ✅ (Premium only) |
| WAF | ✅ (built-in) | ❌ (different service) |
| Load balancing | ✅ (L7) | ❌ |
| URL path routing | ✅ | ❌ |
| Use case | Web app frontend | Network security gateway |

### WAF policies

WAF can run in two modes:
- **Detection**: log attacks but don't block
- **Prevention**: block attacks that match OWASP rules

```bash
# Create WAF policy
az network application-gateway waf-policy create \
  -g rg-prod -n waf-policy --type OWASP --version 3.2

# Set to prevention mode
az network application-gateway waf-policy policy-setting update \
  -g rg-prod --policy-name waf-policy --mode Prevention --state Enabled
```

## Azure Front Door

Global L7 load balancer + CDN + WAF at the edge.

| Feature | Front Door | Application Gateway |
|---------|-----------|---------------------|
| Scope | Global (edge POPs) | Regional |
| CDN | ✅ built-in | ❌ |
| WAF | ✅ | ✅ |
| Private Link origin | ✅ (Premium) | ✅ |
| SSL offload | ✅ | ✅ |
| Use case | Global web apps | Single-region web apps |

**Exam tip**: Front Door WAF rules apply at the **edge** (before traffic reaches your origin). App Gateway WAF rules apply at the **region** level.

## DDoS Protection

```bash
# Create DDoS Protection Plan
az network ddos-protection create -g rg-prod -n ddos-plan

# Associate with VNet
az network vnet update -g rg-prod -n vnet-prod \
  --ddos-protection-plan ddos-plan --ddos-protection true
```

**When to recommend Standard over Basic**:
- You have public IPs that need protection
- You need DDoS attack analytics and metrics
- You want the DDoS Rapid Response (DRR) team
- You need **cost protection** (Azure credits back costs incurred by a DDoS attack)

---
## Summary

| Service | Key exam fact |
|---------|---------------|
| **Azure Firewall** | L3-L7, FQDN filtering, threat intel. Premium adds TLS inspection + IDPS. |
| **Firewall Manager** | Hierarchical policies, secured virtual hubs. |
| **App Gateway** | L7 load balancer + WAF. Regional. |
| **Front Door** | Global L7 + CDN + WAF at edge. Premium for Private Link origins. |
| **WAF** | Detection vs Prevention mode. OWASP rule sets. |
| **DDoS Standard** | Per-VNet, analytics, DRR team, cost protection. |

**Next lab**: [03 — Secure Compute, Storage, and Databases](../../03-compute-storage-databases/)